# Significant TF counts

Bar plot of `# TFs significant per dataset` under two criteria:

- **distance > NC max** — calibration-robust, headline number.
- **pval_mean < 0.05** — parametric, hatched where calibration is anti-conservative.

**Input:** `results/significant_tf_counts/significant_tf_counts.tsv` (from `scripts/combine_count_tables.py`)
**Output:** `results/significant_tf_counts/significant_tf_counts.{pdf,png}`

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

HERE = Path.cwd()
EDIST = HERE if HERE.name == "edist" else HERE.parent
JAMB = EDIST.parents[2]
REPO = JAMB.parents[2]
RESULTS = EDIST / "results" / "significant_tf_counts"
RESULTS.mkdir(parents=True, exist_ok=True)

with open(REPO / "config/colors/production_TF-Perturb-seq.yaml") as f:
    CFG = yaml.safe_load(f)

SHORT = {
    "Hon_WTC11-cardiomyocyte-differentiation_TF-Perturb-seq": "HonCM",
    "Huangfu_HUES8-definitive-endoderm-differentiation_TF-Perturb-seq": "HuangfuDE",
    "Huangfu_HUES8-embryonic-stemcell-differentiation_TF-Perturb-seq": "HuangfuESC",
    "Gersbach_WTC11-hepatocyte-differentiation_TF-Perturb-seq": "GersbachHep",
    "Engreitz_WTC11-endothelial-cells_TF-Perturb-seq": "EngreitzEndo",
}
SHORT_TO_FULL = {v: k for k, v in SHORT.items()}
DATASET_ORDER = ["HonCM", "HuangfuDE", "HuangfuESC", "GersbachHep"]
COLORS = {s: CFG["dataset_colors"][SHORT_TO_FULL[s]] for s in DATASET_ORDER}

print("RESULTS:", RESULTS)
from matplotlib.patches import Patch


The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.


IndexError: 2

In [ ]:
counts = pd.read_csv(RESULTS / "significant_tf_counts.tsv", sep="\t").set_index("dataset_id")
counts = counts.reindex([SHORT_TO_FULL[s] for s in DATASET_ORDER if SHORT_TO_FULL[s] in counts.index])

x = np.arange(len(counts))
w = 0.38
dist = counts["n_sig_distance_gt_NC_max"].to_numpy()
pval = counts["n_sig_pval_lt_0p05"].to_numpy()
colors = [COLORS[SHORT[d]] for d in counts.index]

fig, ax = plt.subplots(figsize=(1.6 * len(counts) + 2, 5))
b_dist = ax.bar(x - w / 2, dist, w, color=colors, edgecolor="black", linewidth=0.6)
b_pval = ax.bar(x + w / 2, pval, w, color=colors, edgecolor="black", linewidth=0.6, alpha=0.55)
for bar, state in zip(b_pval, counts["calibration_state"]):
    if state == "anti-conservative":
        bar.set_hatch("////")
for bar in list(b_dist) + list(b_pval):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([SHORT[d] for d in counts.index])
ax.set_ylabel("# TFs called significant (targeting only)")
ax.set_title("Energy-distance significance per production dataset")
ax.spines[["top", "right"]].set_visible(False)
ax.set_ylim(0, max(dist.max(), pval.max()) * 1.18)
ax.legend(handles=[
    Patch(facecolor="grey", edgecolor="black", label="distance > NC max"),
    Patch(facecolor="grey", edgecolor="black", alpha=0.55, label="pval_mean < 0.05"),
    Patch(facecolor="grey", edgecolor="black", alpha=0.55, hatch="////",
          label="pval_mean (calibration broken)"),
], loc="upper right", frameon=False, fontsize=8)

plt.tight_layout()
fig.savefig(RESULTS / "significant_tf_counts.pdf")
fig.savefig(RESULTS / "significant_tf_counts.png", dpi=200)
plt.show()